In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter, defaultdict
import time
import re

# Set paths
TRAIN_PATH = '../data/train.src.tok'
DEV_PATH = '../data/dev_set.csv'
TEST_PATH_NO_ANSWER = '../data/test_set_no_answer.csv'
TEST_PATH_PRED = '../data/test_set_pred.txt'

In [ ]:
# Load a subset of Training Data for initial development
#TODO: make utils
def load_training_data(filepath, limit=None):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = []
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            lines.append(line.strip())
    return lines

# Load first 200k lines for initial testing (adjust as needed for full training)
LIMIT_LINES = 200000 
train_lines = load_training_data(TRAIN_PATH, limit=LIMIT_LINES)
print(f"Loaded {len(train_lines)} lines from training data.")
print("Sample lines:")
print(train_lines[:3])

Loaded 200000 lines from training data.
Sample lines:
["australia ' s current account deficit shrunk by a record 1 . 11 billion dollars - lrb - 1 . 11 billion us - rrb - in the june quarter due to soaring commodity prices , figures released monday showed .", 'at least two people were killed in a suspected bomb attack on a passenger bus in the strife - torn southern philippines on monday , the military said .', 'australian shares closed down 1 . 1 percent monday following a weak lead from the united states and lower commodity prices , dealers said .']


In [3]:
# Load Dev Set
dev_df = pd.read_csv(DEV_PATH)
print("Dev set shape:", dev_df.shape)
display(dev_df.head())

# Load Test Set (No Answer)
test_no_answer_df = pd.read_csv(TEST_PATH_NO_ANSWER)
print("Test set (no answer) shape:", test_no_answer_df.shape)
display(test_no_answer_df.head())

Dev set shape: (94825, 3)


,context,first letter,answer
0,south korea and the united states on monday wa...,d,day
1,after agreeing to drastically cut its car impo...,t,the
2,three soldiers were injured in a bombing ambus...,m,morning
3,an aviation official says a yemeni investigati...,w,was
4,the movement for democracy in liberia - lrb - ...,t,to


Test set (no answer) shape: (94826, 2)


,context,first letter
0,ukraine and poland agreed to help each other i...,j
1,"richard blumenthal , the longtime attorney gen...",t
2,the leaders of france and britain say the coun...,t
3,""" some of the things i did in 1111 , i ' m bet...",n
4,"the white house , acknowledging that it was st...",","


In [4]:
class NGramModel:
    def __init__(self, n=3):
        self.n = n
        self.counts = defaultdict(Counter)
        self.context_counts = Counter() # Count of times a context appears (for probability calc if needed, though mostly we just need max count here)
    
    def train(self, lines):
        """
        Trains the N-gram model on the list of tokenized strings (lines).
        """
        print(f"Training {self.n}-gram model...")
        start_time = time.time()
        for line in lines: # Iterating through lines (sentences)
            tokens = line.split() # The data is already tokenized with spaces
            
            # Skip if sentence is shorter than n
            if len(tokens) < self.n:
                continue

            # Sliding window to get n-grams
            for i in range(len(tokens) - self.n + 1):
                context = tuple(tokens[i : i + self.n - 1])
                word = tokens[i + self.n - 1]
                self.counts[context][word] += 1
                self.context_counts[context] += 1
                
        print(f"Training finished in {time.time() - start_time:.2f} seconds.")
        print(f"Number of unique contexts: {len(self.counts)}")

    def get_count(self, context):
        return self.counts.get(tuple(context), Counter())

    def get_best_candidates(self, context, first_letter):
        """
        Returns list of (word, count) candidates starting with first_letter
        given the context tuple.
        """
        candidates = self.counts.get(tuple(context), Counter())
        # Filter by first letter
        filtered = [(word, count) for word, count in candidates.items() if str(word).lower().startswith(first_letter.lower())]
        # Sort by count desc
        filtered.sort(key=lambda x: x[1], reverse=True)
        return filtered

class BackoffNGramModel:
    def __init__(self, max_n=5):
        self.models = {}
        self.max_n = max_n
        for n in range(1, max_n + 1):
            self.models[n] = NGramModel(n)

    def train(self, lines):
        for n in range(1, self.max_n + 1):
            self.models[n].train(lines)

    def predict(self, context_str, first_letter):
        # Tokenize context just in case (though input is string)
        # The context in csv is a string. "word1 word2 word3 ..."
        context_tokens = str(context_str).split()
        
        # Try from max_n down to 1
        for n in range(self.max_n, 1, -1):
            # For N-gram prediction, we need the last N-1 words of context
            needed_context_len = n - 1
            
            # If context is too short for this N, skip to lower N
            if len(context_tokens) < needed_context_len:
                continue
            
            # Get the last N-1 tokens as context
            if needed_context_len > 0:
                current_context = tuple(context_tokens[-needed_context_len:])
            else:
                 current_context = () # Should handle bigram context being 1 word, wait n=2 -> context len 1. n=1 -> context len 0.
            
            candidates = self.models[n].get_best_candidates(current_context, first_letter)
            
            if candidates:
                return candidates[0][0] # Return best match
        
        # If still no match, look at unigram (1-gram) model
        # 1-gram model (N=1) context is empty tuple () 
        # But wait, 1-gram usually implies P(w). My NGramModel implementation for n=1:
        # loop range(len - 0): context=tokens[i:i], word=tokens[i]. context is indeed ().
        candidates = self.models[1].get_best_candidates((), first_letter)
        if candidates:
            return candidates[0][0]
            
        return "" # Should not happen ideally if unigram covers all vocab check

In [5]:
# Initialize Backoff Model (Up to 3-gram for quick testing, but we can do 5)
MAX_N = 3 # Start with 3 for speed in dev, increase to 5 for final
backoff_model = BackoffNGramModel(max_n=MAX_N)

# Train on the subset
backoff_model.train(train_lines)

Training 1-gram model...
Training finished in 2.69 seconds.
Number of unique contexts: 1
Training 2-gram model...
Training finished in 4.43 seconds.
Number of unique contexts: 52578
Training 3-gram model...
Training finished in 8.85 seconds.
Number of unique contexts: 876688


In [ ]:
# Evaluation Function
# TODO: make utils
def evaluate_model(model, df):
    correct = 0
    total = len(df)
    
    start_eval = time.time()
    for index, row in df.iterrows():
        context = row['context']
        first_letter = row['first letter']
        true_answer = row['answer']
        
        prediction = model.predict(context, first_letter)
        
        if prediction == true_answer:
            correct += 1
            
    accuracy = correct / total
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Evaluation time: {time.time() - start_eval:.2f}s")
    return accuracy

# Run evaluation on dev set
print(f"Evaluating {MAX_N}-gram backoff model on {len(dev_df)} dev examples...")
evaluate_model(backoff_model, dev_df)

Evaluating 3-gram backoff model on 94825 dev examples...
Accuracy: 0.5191
Evaluation time: 44.94s


0.5191247034010018

In [ ]:
# Generate Predictions for Test Set
# TODO: make utils
def generate_test_predictions(model, test_df, output_path):
    print(f"Generating predictions for {len(test_df)} test examples...")
    predictions = []
    start_time = time.time()
    
    for index, row in test_df.iterrows():
        context = row['context']
        first_letter = row['first letter']
        
        # Predict
        prediction = model.predict(context, first_letter)
        
        # If no prediction found (empty string), we might need a fallback or just empty
        # The instructions say "if the trigram context is not found... return ' '"
        # But my predict function returns "" if nothing found.
        # However, for submission we probably want a word?
        # If empty, let's output a placeholder or just empty string.
        # Actually backoff should usually find something (unigram).
        
        predictions.append(prediction)
        
    print(f"Prediction finished in {time.time() - start_time:.2f} s")
    
    # Save to file
    with open(output_path, 'w', encoding='utf-8') as f:
        for pred in predictions:
            f.write(f"{pred}\n")
    print(f"Saved predictions to {output_path}")

# Run generation
generate_test_predictions(backoff_model, test_no_answer_df, TEST_PATH_PRED)

Generating predictions for 94826 test examples...
Prediction finished in 47.44 s
Saved predictions to ../data/test_set_pred.txt


In [9]:
# Run check.py
# check.py expects to be run from its directory or relative paths might be off.
# check.py is in src/ and looks for ../data/test_set_pred.txt
# If we run it from notebooks/ via python command, we need to be careful with CWD.

# Let's run it via shell command for simplicity
!python ../src/check.py

The number of rows in the DataFrame matches the number of lines in the text file.
